In [1]:
%pip install -q otter-grader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.5/145.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 8.3 MB/s eta 0:00:00


In [5]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/01-InClass-Exercise-What-Is-DS-A1/tests/"
    for _q in ("q1", "q2", "q3", "q4", "q5"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")


In [6]:
# Initialize Otter
import otter
grader = otter.Notebook()

# In-Class Exercise 01: What is Data Science? — question types, then Pandas

**DS701 — Session 2 (Wed Sep 9, 2026)**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/01-InClass-Exercise-What-Is-DS-A1/01-InClass-Exercise-What-Is-DS-A1.ipynb)

**Time: ~45 minutes** (today shares the session with the Spark pitches). Work in groups of 2–3.

Parts marked **(autograded)** are submitted to Gradescope; open-ended parts are graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked — staff will stop by your table.

**Plan**

| Part | Topic | Grading | Time |
|---|---|---|---|
| 1 | Classify the question: ten scenarios | autograded + short written justifications | ~12 min |
| 2 | Spot the type error | autograded | ~5 min |
| 3 | **Stretch:** Pandas on a real dataset — read, inspect, filter, `groupby`, plot | autograded + a plot | ~20 min |
| 4 | Which question did your `groupby` answer? | written, participation | ~8 min |

Part 3 is the *stretch* section: the lecture only mentioned that "clean and explore" eats
half of a project; here you do a first pass of it in Pandas. If Pandas is new to you,
keep the [Pandas tutorial](https://tools4ds.github.io/ds701/A3-Pandas.html)
open — each task below points at the section it draws on.

Dependencies: `pandas`, `numpy`, `matplotlib`.

## Setup

Run the next three cells as they are. Nothing to fill in yet.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
# Lecture section. The build script sets this per variant (A1 / B1); the
# activity branches on it below so the two sections get different scenarios,
# data slices, and hidden-test values while sharing one master and one skeleton.
SECTION = "A1"
SEED = {"A1": 701, "B1": 702}[SECTION]
rng = np.random.default_rng(SEED)
print("Section:", SECTION)

Section: A1


In [9]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/01-InClass-Exercise-What-Is-DS-A1/tests/"
    for _q in ("q1", "q2", "q3", "q4", "q5"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

## Part 1: classify the question

The lecture's taxonomy (after Leek & Peng, 2015), from least to most demanding:

| Type | Asks | Standard of evidence |
|---|---|---|
| `descriptive` | What does the data say, summarized? | the numbers are computed correctly |
| `exploratory` | What patterns are worth a closer look? | a pattern is there to be seen — no claim it generalizes |
| `inferential` | Does the pattern hold beyond this sample? | sampling uncertainty is quantified |
| `predictive` | What will happen for a new case? | held-out accuracy |
| `causal` | What happens if we *intervene*? | confounding is controlled (ideally by randomization) |
| `mechanistic` | Exactly how does the effect operate? | the mechanism is established, usually deterministically |

**Reading the scenarios.** Ask two things of each one: *is it about this data only, or about
a population / a new case?* and *does it ask what would happen if someone acted, or how
something works?* The verbs matter — "report", "summarize" (descriptive); "look for",
"see what emerges" (exploratory); "in general", "across all", "estimate ... with a margin
of error" (inferential); "guess", "will this one", "probability that this ..." (predictive);
"would", "cause", "because of", "if we changed" (causal); "how exactly", "through which
pathway" (mechanistic).

Run the next cell to print **your section's ten scenarios**. Sections A1 and B1 have
different lists.

In [10]:
POOL = {
    1: "The city publishes the average number of bike-share rides per day for each month of 2025.",
    2: "A hospital wants to know whether patients given a new discharge checklist are readmitted less often *because of* the checklist.",
    3: "A streaming service wants to guess which of its 40 million current subscribers will cancel next month.",
    4: "A biologist wants to know exactly which cellular pathway a drug uses to lower blood pressure.",
    5: "A polling firm surveys 1,200 voters and asks whether support for a ballot measure differs between age groups in the state as a whole.",
    6: "An analyst plots every column of a new sales dataset against every other to see what might be worth digging into.",
    7: "A retailer wants to know whether sending a coupon by text message instead of by email would increase how many people redeem it.",
    8: "A university reports the median starting salary of last year's graduates, by major.",
    9: "An insurer wants the probability that a specific new applicant will file a claim in the next twelve months.",
    10: "A study of 500 restaurants asks whether higher Yelp ratings are associated with higher revenue among restaurants in general.",
    11: "A materials scientist wants to explain, step by step, how a particular alloy conducts heat.",
    12: "A data journalist digs through a leaked dataset of flight delays looking for anything surprising, without a specific hypothesis in mind.",
    13: "A ride-share company reports its total revenue and total number of trips for the second quarter.",
    14: "A model reads a chest X-ray and outputs whether this patient likely has pneumonia.",
    15: "A researcher clusters 10,000 news articles to see what topics emerge.",
    16: "An economist asks whether raising the minimum wage in a county would reduce employment there.",
    17: "From a random sample of 2,000 households, estimate the fraction of all households in the state that have broadband, with a margin of error.",
    18: "Determine how a specific mutation changes the shape of a protein so that it no longer binds its receptor.",
    19: "A bank wants to know whether waiving overdraft fees would lead customers to keep higher balances.",
    20: "HR wants every current employee ranked by how likely they are to resign in the next six months.",
}

# Which pool items each section sees, and in what order.
SCENARIO_IDS = {
    "A1": [3, 8, 16, 6, 9, 1, 5, 4, 10, 2],
    "B1": [13, 19, 15, 14, 17, 11, 7, 12, 20, 18],
}[SECTION]

SCENARIOS = {i + 1: POOL[pid] for i, pid in enumerate(SCENARIO_IDS)}
TYPES = ["descriptive", "exploratory", "inferential", "predictive", "causal", "mechanistic"]

for num, text in SCENARIOS.items():
    print(f"{num:>2}. {text}")

 1. A streaming service wants to guess which of its 40 million current subscribers will cancel next month.
 2. A university reports the median starting salary of last year's graduates, by major.
 3. An economist asks whether raising the minimum wage in a county would reduce employment there.
 4. An analyst plots every column of a new sales dataset against every other to see what might be worth digging into.
 5. An insurer wants the probability that a specific new applicant will file a claim in the next twelve months.
 6. The city publishes the average number of bike-share rides per day for each month of 2025.
 7. A polling firm surveys 1,200 voters and asks whether support for a ballot measure differs between age groups in the state as a whole.
 8. A biologist wants to know exactly which cellular pathway a drug uses to lower blood pressure.
 9. A study of 500 restaurants asks whether higher Yelp ratings are associated with higher revenue among restaurants in general.
10. A hospital wan

### Part 1a (autograded): the classification

**Task.** Fill in `classification` so that `classification[n]` is the question type of
scenario `n`, as one of the six lower-case strings in `TYPES`. All ten scenarios must be
classified.

Talk each one through in your group before typing — the point is the *reason*, which
you write down in Part 1b.

In [14]:
classification = {
    1: "predictive",
    2: "descriptive",
    3: "causal",
    4: "exploratory",
    5: "predictive",
    6: "descriptive",
    7: "inferential",
    8: "mechanistic",
    9: "inferential",
    10: "causal",
}

# Prints your answers next to the scenarios so you can double-check them.
for num, text in SCENARIOS.items():
    print(f"{num:>2}. [{classification.get(num, '???'):<12}] {text[:80]}...")

 1. [predictive  ] A streaming service wants to guess which of its 40 million current subscribers w...
 2. [descriptive ] A university reports the median starting salary of last year's graduates, by maj...
 3. [causal      ] An economist asks whether raising the minimum wage in a county would reduce empl...
 4. [exploratory ] An analyst plots every column of a new sales dataset against every other to see ...
 5. [predictive  ] An insurer wants the probability that a specific new applicant will file a claim...
 6. [descriptive ] The city publishes the average number of bike-share rides per day for each month...
 7. [inferential ] A polling firm surveys 1,200 voters and asks whether support for a ballot measur...
 8. [mechanistic ] A biologist wants to know exactly which cellular pathway a drug uses to lower bl...
 9. [inferential ] A study of 500 restaurants asks whether higher Yelp ratings are associated with ...
10. [causal      ] A hospital wants to know whether patients given a new 

In [15]:
grader.check("q1")

q1 results: All test cases passed!

<!-- BEGIN QUESTION -->

### Part 1b (written): one-line justifications

For **each** of the ten scenarios, write one line: the type you chose and the word or
phrase in the scenario that decided it. Where your group disagreed, say what the
alternative reading was.

Format (one line per scenario, in the cell below):

```
1. predictive — "guess which ... will cancel next month": a new case, judged by accuracy
2. ...
```

> Be ready to defend any one of these when staff visit your table.

1.

<!-- END QUESTION -->

## Part 2 (autograded): spot the type error

The lecture's claim: most real analyses fail by **answering a different question type
than was asked**. Read your section's case and fill in two strings.

In [ ]:
CASE = {
    "A1": """A hospital's leadership asks: 'Would adding more chaplain visits shorten
patients' hospital stays?' The analytics team fits a model that predicts length of stay
from forty features. Chaplain visits get high feature importance and the held-out
error is excellent, so the team recommends hiring more chaplains.""",
    "B1": """A school district asks: 'Would giving every student the tutoring app raise
test scores?' The analytics team builds a dashboard showing that students who already
use the app score, on average, 12 points higher than those who do not, and recommends
buying licenses for everyone.""",
}[SECTION]
print(CASE)

**Task.** Set `asked` to the question type leadership actually asked, and `answered` to
the question type the team's analysis actually answers. Use the same six lower-case
strings.

In [ ]:
asked = ...
answered = ...
print("asked:", asked, "| answered:", answered)

In [ ]:
grader.check("q2")

## Part 3 (stretch): a first pass at "clean and explore", in Pandas

The lecture said stage 3 of the lifecycle — clean and explore — is where half the effort
goes, and "plots before models, always". Here you do the first 20 minutes of that stage on
a small, real dataset: the 891 passengers in the Kaggle **Titanic** training set.

| Column | Meaning |
|---|---|
| `Survived` | 0 = no, 1 = yes |
| `Pclass` | ticket class: 1, 2, 3 (a proxy for socio-economic status) |
| `Sex`, `Age` | as recorded; `Age` is missing for some passengers |
| `SibSp`, `Parch` | siblings/spouses and parents/children aboard |
| `Fare` | ticket price |
| `Embarked` | port: C = Cherbourg, Q = Queenstown, S = Southampton |

Each section works the same skeleton on a **different slice and different columns**, so
your numbers will not match the other section's.

The cell below reads the CSV (`pd.read_csv`; tutorial: [Fetching, storing and retrieving
your data](https://tools4ds.github.io/ds701/A3-Pandas.html#fetching-storing-and-retrieving-your-data)).
It uses the copy in the course repo if you are running there, and otherwise downloads a
byte-identical public copy.

In [ ]:
CANDIDATES = ["../../data/titanic/train.csv", "data/titanic/train.csv"]
URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
path = next((p for p in CANDIDATES if os.path.exists(p)), URL)
df = pd.read_csv(path)
print("loaded from:", path)

# Section-specific choices used throughout Part 3.
GROUP_COL = {"A1": "Pclass", "B1": "Sex"}[SECTION]      # what we group by
VALUE_COL = {"A1": "Age", "B1": "Fare"}[SECTION]        # what we summarize
FILTER_DESC = {"A1": "female passengers", "B1": "third-class passengers"}[SECTION]
print(f"Section {SECTION}: group by {GROUP_COL!r}, summarize {VALUE_COL!r}, filter to {FILTER_DESC}")

df.head()

### Part 3a (autograded): inspect

Before computing anything, look. Run `df.info()` and `df.describe()` in a scratch cell
(tutorial: [Pandas](https://tools4ds.github.io/ds701/A3-Pandas.html#pandas)
— `.head()`, `.info()`, `.describe()`), then fill in:

- `n_rows`, `n_cols` — the shape of `df` (as plain `int`s),
- `n_missing_age` — how many passengers have no recorded `Age` (hint: `.isna().sum()`),
- `value_col_mean` — the mean of your section's `VALUE_COL`, as a `float`.

Notice what `.describe()` does *not* show you: the text columns, and the missing values.

In [ ]:
n_rows = ...
n_cols = ...
n_missing_age = ...
value_col_mean = ...
print(n_rows, n_cols, n_missing_age, round(value_col_mean, 3))

In [ ]:
grader.check("q3")

### Part 3b (autograded): select and filter

Tutorial: [Filtering](https://tools4ds.github.io/ds701/A3-Pandas.html#filtering)
and [Working with data columns](https://tools4ds.github.io/ds701/A3-Pandas.html#working-with-data-columns).

A boolean mask — a `Series` of `True`/`False` the same length as `df` — selects rows:
`df[df["Age"] < 18]` is the passengers under 18.

**Task.** Build `subset`, the rows of `df` for your section's `FILTER_DESC`
(printed above: A1 → `Sex == "female"`, B1 → `Pclass == 3`). Then compute

- `n_subset` — how many rows it has (`int`),
- `subset_survival_rate` — the mean of `Survived` in the subset (`float`).
  Since `Survived` is 0/1, its mean *is* the survival rate.

In [ ]:
mask = ...
subset = ...
n_subset = ...
subset_survival_rate = ...
print(f"{FILTER_DESC}: {n_subset} rows, survival rate {subset_survival_rate:.3f}")
subset.head()

In [ ]:
grader.check("q4")

### Part 3c (autograded): `groupby` and aggregate

Tutorial: [Understanding pandas DataFrame.groupby()](https://tools4ds.github.io/ds701/A3-Pandas.html#understanding-pandas-dataframe.groupby)
— *split-apply-combine*: split the rows into groups by a column, apply an aggregate to each
group, combine the results into one table.

**Task.** Using the **full** `df` (not `subset`):

1. `by_group` — group by `GROUP_COL` and aggregate `VALUE_COL` with **both** `"mean"` and
   `"count"` (hint: `.agg(["mean", "count"])`). The result is a DataFrame with one row per
   group and columns `mean` and `count`.
2. `survival_by_group` — group by `GROUP_COL` and take the mean of `Survived` (a Series:
   the survival rate per group).
3. `top_group` — the group label with the **highest** survival rate (hint: `.idxmax()`,
   or `.sort_values()` and read off the first index).

Then look at `by_group["count"]`: it tells you how much data sits behind each mean.

In [ ]:
by_group = ...
survival_by_group = ...
top_group = ...
print(by_group)
print()
print(survival_by_group)
print("\nhighest survival rate:", top_group)

In [ ]:
grader.check("q5")

<!-- BEGIN QUESTION -->

### Part 3d (participation): one plot

Tutorial: [Plotting methods](https://tools4ds.github.io/ds701/A3-Pandas.html#plotting-methods)
and [Visualizing Grouped Data](https://tools4ds.github.io/ds701/A3-Pandas.html#visualizing-grouped-data).

**Task.** Make a bar chart of `survival_by_group` (`.plot(kind="bar")` on a Series does
it in one line). Give it a title and a y-axis label, and add a horizontal line at the
overall survival rate `df["Survived"].mean()` so each bar can be read against the whole
ship. Then, in the markdown cell that follows, write one sentence describing what the
plot shows.

In [ ]:
# ax = survival_by_group.plot(kind="bar", ...)      # the bars
# ax.axhline(df["Survived"].mean(), ...)            # overall rate, for reference
# ax.set_ylabel(...); ax.set_title(...)
# plt.show()
...

**What the plot shows (one sentence):**

*(write here)*

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## Part 4 (written, participation): which question did your `groupby` answer?

Tie the two halves together. In the cell below, answer in a short paragraph:

1. **Which question type** from Part 1's taxonomy does your `survival_by_group` result
   answer, and *why that one and not the next one up*? (Look back at the "standard of
   evidence" column.)
2. A journalist reads your table and writes: *"Being in `top_group` **made** passengers
   more likely to survive."* Which type is that claim? Name one thing — a variable in this
   dataset or one that is not in it — that could explain the association without the
   journalist's claim being true.
3. What would you need (more data? a different design? an assumption?) to move your
   result one rung up the ladder toward the journalist's claim?

> Be ready to explain your reasoning if your group is visited — staff will ask exactly
> these questions at your table.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Wrap-up

Before you submit, make sure the whole notebook runs top to bottom (*Runtime → Restart
and run all* on Colab) and that Parts 1–3 pass their public tests. Submit the `.ipynb`
to Gradescope; the hidden tests use your section's numbers, so a notebook copied from the
other section will fail visibly.

What you practiced today: **classify the question before choosing the technique**, and
the first pass of **clean and explore** — read, inspect, filter, group, plot — that every
project in this course will start with.